# TweetAge v10: Kneser-Ney modificado sobre caracteres, palabras y raíces

v10 añade un tercer modelo de lenguaje por clase, sobre **raíces** (Snowball de NLTK en español), al ensamble de v6 (caracteres de orden 4 y palabras de orden 3, ambos con Kneser-Ney modificado). Los pesos del ensamble se eligen en un simplex de paso 0,25 dentro de cada pliegue, con una partición interna. La validación exterior no interviene en esa elección.

**Reglas fijas** (ver `claude/DECISION_REGLAS_TWEETAGE.md`; al crear el notebook ese documento no estaba en el repositorio, así que las reglas se transcriben del pedido):

- Solo modelos probabilísticos de lenguaje (KN modificado) entrenados por el equipo sobre caracteres, palabras y raíces.
- Quedan prohibidos los embeddings, Word2Vec, GloVe, TF-IDF, cualquier vectorización del texto, los modelos preentrenados y LSTM/GRU.
- El stemmer es `SnowballStemmer('spanish')` de NLTK: un algoritmo de reglas que no necesita `nltk.download`. No se usan spaCy ni lematizadores.
- La semilla 42 se fija en `random`, `numpy` y `torch`. Torch solo se importa para fijar la semilla.
- No se regenera ninguna partición: se usa `split_idx_v6_seed42.npz`.
- **Regla de decisión (la de v6):** la solución pasa solo si la media de F1(v10) − F1(v6) en los cinco pliegues es positiva y además mayor que su desviación muestral (ddof=1).
- El holdout se evalúa una sola vez, y solo si la regla se cumple.

**Ejecución:** Google Colab, solo CPU (KN es Python puro y no usa GPU). Los datos se leen de Drive y los resultados se escriben en `RESULTS/tweetage_v10_raices/`. Cada pliegue y cada modelo se guardan en caché con una huella. Si una sesión se corta, basta con volver a ejecutar todo: los pliegues terminados se recuperan sin reentrenar, y si una huella no coincide la ejecución se detiene.

## 1. Configuración (única celda con rutas)

In [ ]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import random
import re
import resource
import sys
import time
import unicodedata
from collections import Counter
from datetime import datetime
from functools import lru_cache
from importlib.metadata import version as package_version
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import torch
from nltk.stem.snowball import SnowballStemmer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython import get_ipython
from IPython.display import display, Markdown

# Rutas: solo aqui. Nada local, nada de files.upload() ni rutas relativas.
DATA_DIR = Path('/content/drive/MyDrive/PLN/tweet-age-2026-puj')
RESULTS_DIR = Path('/content/drive/MyDrive/PLN/RESULTS')
OUT_DIR = RESULTS_DIR / 'tweetage_v10_raices'
SUBMISSION_PATH = RESULTS_DIR / 'tweetage_v10_submission.csv'
TRAIN_FILENAME = 'age_train.csv'
TEST_FILENAME = 'age_test.csv'
SAMPLE_FILENAME = 'age_submission.csv'
SPLIT_FILENAME = 'split_idx_v6_seed42.npz'
# Opcional: si los notebooks originales estan en Drive, el gate AST tambien compara contra ellos.
ORIGINAL_NOTEBOOKS = {
    'v6': Path('/content/drive/MyDrive/PLN/TweetAge_v6_charLM_kn.ipynb'),
    'v9': Path('/content/drive/MyDrive/PLN/TweetAge_v9_LSTM_LM.ipynb'),
}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASSES = np.array(['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'])
BOS, EOS, UNK = '\x02', '\x03', '\x01'
BASE_EPSILON = 1e-6
MODELS = ['char', 'word', 'root']
MODEL_NAMES = {'char': 'caracteres', 'word': 'palabras', 'root': 'raices'}
ORDERS = {'char': 4, 'word': 3, 'root': 3}
ENCODER_LEVEL = {'char': 'char', 'word': 'word', 'root': 'word'}
V6_WEIGHTS = (0.75, 0.25, 0.0)
SIMPLEX = [(i / 4, j / 4, (4 - i - j) / 4) for i in range(4, -1, -1) for j in range(4 - i, -1, -1)]
PARAMS = dict(inner_fraction=0.1)  # nombre que usa inner_split, copiada de v9
EXPECTED_EXCLUDED = 155
EXPECTED_FOLD_SIZES = [26519, 26654, 26979, 26627, 26286]
V6_FOLD1_F1 = 0.443844
GATE_TOLERANCE = 1e-6
IMPLEMENTATION_VERSION = 'v10_raices_kn_20261016_v1'
STEMMER = SnowballStemmer('spanish')

print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__, '| nltk', package_version('nltk'), '| torch', torch.__version__)
print('Semilla', SEED, 'en random, numpy y torch. Computo solo en CPU.')
print('Ordenes:', ORDERS, '| Combinaciones del simplex:', len(SIMPLEX))
print('Entradas:', DATA_DIR)
print('Salidas:', OUT_DIR)
print('Submission condicional:', SUBMISSION_PATH)

## 2. Drive y entradas
Se monta Drive y se comprueba que existan los cuatro archivos de entrada y la carpeta de salida (se crea si no existe). Si falta algo, la ejecución se detiene. A partir de aquí todo queda registrado en `ejecucion.log`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def verify_drive_inputs():
    required = {
        'train': DATA_DIR / TRAIN_FILENAME,
        'test': DATA_DIR / TEST_FILENAME,
        'sample': DATA_DIR / SAMPLE_FILENAME,
        'split': DATA_DIR / SPLIT_FILENAME,
    }
    for label, path in required.items():
        print(label, ':', path, 'OK' if path.is_file() else 'FALTA')
    missing = [str(path) for path in required.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan archivos de entrada en Drive:\n' + '\n'.join(missing) + '\nSubalos a DATA_DIR. No se crea ninguna particion nueva y no se continua.')
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    if not OUT_DIR.is_dir():
        raise NotADirectoryError(f'No se pudo crear la carpeta de salida {OUT_DIR}.')
    print('Carpeta de salida lista:', OUT_DIR)
    return required

INPUT_PATHS = verify_drive_inputs()
TRAIN_PATH, TEST_PATH, SAMPLE_PATH, SPLIT_PATH = (INPUT_PATHS[name] for name in ['train', 'test', 'sample', 'split'])
LOG_PATH = OUT_DIR / 'ejecucion.log'

def log(*parts):
    message = ' '.join(str(part) for part in parts)
    print(message, flush=True)
    with LOG_PATH.open('a', encoding='utf-8') as handle:
        handle.write(f"{datetime.now().isoformat(timespec='seconds')} {message}\n")

def memory_report():
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_mib = peak / 1048576 if sys.platform == 'darwin' else peak / 1024
    try:
        import psutil
        current_mib = psutil.Process().memory_info().rss / 1048576
    except ImportError:
        current_mib = float('nan')
    return f'memoria RSS actual {current_mib:.0f} MiB, pico del proceso {peak_mib:.0f} MiB'

log('=== Inicio', IMPLEMENTATION_VERSION, '|', memory_report())

## 3. Lectura, esquema y hashes
Train y test usan punto y coma con BOM; la plantilla usa coma sin BOM (formato confirmado en v6). Si las columnas no son exactamente las esperadas, se muestran las detectadas y se detiene. El SHA-256 de `age_train.csv` tiene que coincidir con `train_sha256` del NPZ.

In [ ]:
def read_raw(path, label):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first = prefix.decode('utf-8-sig').splitlines()[0]
    separator = csv.Sniffer().sniff(first, delimiters=',;\t|').delimiter
    has_bom = prefix.startswith(b'\xef\xbb\xbf')
    expected_separator = ';' if label in {'train', 'test'} else ','
    expected_bom = label in {'train', 'test'}
    if separator != expected_separator or has_bom != expected_bom:
        raise ValueError(f'Formato inesperado en {path.name}: separador={separator!r}, BOM={has_bom}. Esperado: {expected_separator!r}, BOM={expected_bom}.')
    return pd.read_csv(path, sep=separator, encoding='utf-8-sig', keep_default_na=False, dtype=str)

def confirm_schema(frame, label, required):
    actual = frame.columns.tolist()
    print(label, '| columnas detectadas:', actual, '| esperadas:', required)
    if actual != required:
        raise ValueError(f'Esquema inesperado en {label}. Columnas detectadas: {actual}; esperadas: {required}. Se detiene.')
    return frame

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

train_raw = confirm_schema(read_raw(TRAIN_PATH, 'train'), 'train', ['id', 'text', 'age_range'])
test_raw = confirm_schema(read_raw(TEST_PATH, 'test'), 'test', ['id', 'text'])
sample = confirm_schema(read_raw(SAMPLE_PATH, 'sample'), 'sample', ['id', 'age_range'])
assert set(train_raw.age_range) == set(CLASSES), 'Las etiquetas no son los seis rangos esperados.'
assert train_raw.id.is_unique and test_raw.id.is_unique and sample.id.is_unique
assert set(sample.id) == set(test_raw.id) and len(sample) == len(test_raw), 'Los IDs de la plantilla y del test no coinciden.'

with np.load(SPLIT_PATH, allow_pickle=False) as saved:
    split = {name: saved[name].copy() for name in saved.files}
INPUT_SHA = {name: sha256_file(path) for name, path in INPUT_PATHS.items()}
TRAIN_SHA, TEST_SHA, SPLIT_SHA = INPUT_SHA['train'], INPUT_SHA['test'], INPUT_SHA['split']
if str(split['train_sha256']) != TRAIN_SHA:
    raise RuntimeError(f"El SHA-256 de age_train.csv ({TRAIN_SHA}) no coincide con el del NPZ ({split['train_sha256']}). No se reutilizan indices ni se crea otra particion.")
assert int(split['seed']) == SEED

log('Filas train/test/plantilla:', len(train_raw), len(test_raw), len(sample))
for name, digest in INPUT_SHA.items():
    log('SHA-256', name, digest)
log('Hash del train coincide con train_sha256 del NPZ de v6.')

## 4. Normalización (copiada de v6 sin cambios)
`URL_RE`, `MENTION_RE`, `REPEAT_RE`, `replace_url` y `normalize_text` están copiadas de v6 tal cual. El orden es: decodificar HTML, URL a `<url>`, mención a `<user>`, repeticiones de letras a tres, y descartar vacíos y `#¿NOMBRE?`. **Gate:** el `ast.dump` de cada definición de esta celda tiene que ser idéntico al de v6.

In [ ]:
def current_cell_source():
    shell = get_ipython()
    if shell is None:
        raise RuntimeError('Ejecutar en Colab/IPython: el gate AST lee el codigo de la celda en ejecucion.')
    return shell.history_manager.input_hist_raw[-1]

def top_level_nodes(source):
    nodes = {}
    for node in ast.parse(source).body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            nodes.setdefault(node.name, node)
        elif isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            nodes.setdefault(node.targets[0].id, node)
    return nodes

def ast_gate(source, origin, names):
    current = top_level_nodes(source)
    rows = []
    original = ORIGINAL_NOTEBOOKS.get(origin)
    original_nodes = None
    if original is not None and original.is_file():
        original_nodes = {}
        for cell in json.loads(original.read_text(encoding='utf-8'))['cells']:
            if cell['cell_type'] == 'code':
                for name, node in top_level_nodes(''.join(cell['source'])).items():
                    original_nodes.setdefault(name, node)
    for name in names:
        if name not in current:
            raise RuntimeError(f'Gate AST: {name} no esta definido en esta celda.')
        row = dict(nombre=name, origen=origin, identico_referencia=ast.dump(current[name]) == ast.dump(ast.parse(REFERENCES[origin][name]).body[0]))
        if original_nodes is not None:
            row['identico_notebook_original'] = name in original_nodes and ast.dump(current[name]) == ast.dump(original_nodes[name])
        rows.append(row)
    table = pd.DataFrame(rows)
    display(table)
    if not table.drop(columns=['nombre', 'origen']).to_numpy().all():
        raise RuntimeError(f'Gate AST fallido: alguna definicion difiere de {origin}. No reescribir codigo copiado.')
    log('Gate AST superado para', origin, names, '| notebook original', 'comparado' if original_nodes is not None else 'no disponible en Drive; se usa la referencia literal')
    return True

# Fuente literal de v6 y v9, extraida de los notebooks con ast.get_source_segment.
REFERENCES = {'v6': {'URL_RE': "URL_RE = re.compile(r'https?://[^\\s]+|www\\.[^\\s]+', re.IGNORECASE)",
        'MENTION_RE': "MENTION_RE = re.compile(r'@\\w+', re.UNICODE)",
        'REPEAT_RE': "REPEAT_RE = re.compile(r'([^\\W\\d_])\\1{3,}', re.UNICODE)",
        'replace_url': 'def replace_url(match):\n'
                       '    value = match.group(0)\n'
                       "    core = value.rstrip('.,!?;:)]}')\n"
                       "    return '<url>' + value[len(core):]",
        'normalize_text': 'def normalize_text(text):\n'
                          '    return REPEAT_RE.sub(lambda match: match.group(1) * 3, '
                          "MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))",
        'TOKEN_RE': "TOKEN_RE = re.compile(r'<url>|<user>|\\w+|[^\\w\\s]', re.UNICODE)",
        'tokenize': 'def tokenize(text, level):\n'
                    "    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())",
        'make_encoder': 'def make_encoder(texts, level):\n'
                        '    frequencies = Counter()\n'
                        '    for text in texts:\n'
                        '        frequencies.update(tokenize(text, level))\n'
                        "    cutoff = 1 if level == 'char' else 2\n"
                        '    return {token: chr(index + 256) for index, token in enumerate(sorted(token for '
                        'token, count in frequencies.items() if count >= cutoff))}',
        'encode_text': 'def encode_text(text, level, mapping):\n'
                       "    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))",
        'estimate_discounts': 'def estimate_discounts(counts):\n'
                              '    histogram = Counter(counts.values())\n'
                              '    count1, count2, count3, count4 = [histogram.get(value, 0) for value in '
                              '[1, 2, 3, 4]]\n'
                              '    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0\n'
                              '    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75\n'
                              '    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75\n'
                              '    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 '
                              'else 0.75\n'
                              '    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, '
                              'value in enumerate([first, second, third], 1))\n'
                              '    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)',
        'discount_for': 'def discount_for(count, discounts):\n'
                        '    return discounts[min(count, 3) - 1] if count > 0 else 0.0',
        'ModifiedKN': 'class ModifiedKN:\n'
                      '    def __init__(self, max_order, support):\n'
                      '        self.max_order = max_order\n'
                      '        self.support = set(support) | {EOS, UNK}\n'
                      '\n'
                      '    def fit(self, sequences):\n'
                      '        highest = max(2, self.max_order)\n'
                      '        self.raw = {order: Counter() for order in range(2, highest + 1)}\n'
                      '        for sequence in sequences:\n'
                      '            padded = BOS * (highest - 1) + sequence + EOS\n'
                      '            for order, counts in self.raw.items():\n'
                      '                counts.update(padded[position - order + 1:position + 1] for position '
                      'in range(highest - 1, len(padded)))\n'
                      '        self.continuation = {}\n'
                      '        for order in range(1, highest):\n'
                      '            effective = Counter()\n'
                      '            for gram in self.raw[order + 1]:\n'
                      '                effective[gram[1:]] += 1\n'
                      '            self.continuation[order] = effective\n'
                      '        self.base = self.continuation[1]\n'
                      '        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)\n'
                      '        self.discount_records = []\n'
                      '        def prepare(count_tables, origin):\n'
                      '            discounts_by_order, histories_by_order = {}, {}\n'
                      '            for order, counts in count_tables.items():\n'
                      '                if order == 1:\n'
                      '                    continue\n'
                      '                discounts, histogram = estimate_discounts(counts)\n'
                      '                discounts_by_order[order] = discounts\n'
                      '                totals, removed = Counter(), Counter()\n'
                      '                for gram, count in counts.items():\n'
                      '                    totals[gram[:-1]] += count\n'
                      '                    removed[gram[:-1]] += discount_for(count, discounts)\n'
                      '                histories_by_order[order] = {history: (total, removed[history]) for '
                      'history, total in totals.items()}\n'
                      '                self.discount_records.append(dict(origen=origin, orden=order, '
                      'D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))\n'
                      '            return discounts_by_order, histories_by_order\n'
                      "        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')\n"
                      '        self.cont_discounts, self.cont_hist = prepare(self.continuation, '
                      "'continuaciones')\n"
                      '        self.lower_cached = lru_cache(maxsize=200000)(self._lower)\n'
                      '        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)\n'
                      '        return self\n'
                      '\n'
                      '    def _lower(self, gram):\n'
                      '        if len(gram) == 1:\n'
                      '            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total\n'
                      '        order = len(gram)\n'
                      '        lower = self.lower_cached(gram[1:])\n'
                      '        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))\n'
                      '        if not total:\n'
                      '            return lower\n'
                      '        count = self.continuation[order].get(gram, 0)\n'
                      '        return (count - discount_for(count, self.cont_discounts[order]) + removed * '
                      'lower) / total\n'
                      '\n'
                      '    def probability(self, gram):\n'
                      '        if len(gram) == 1:\n'
                      '            return self.lower_cached(gram)\n'
                      '        order = len(gram)\n'
                      '        lower = self.lower_cached(gram[1:])\n'
                      '        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))\n'
                      '        if not total:\n'
                      '            return lower\n'
                      '        count = self.raw[order].get(gram, 0)\n'
                      '        return (count - discount_for(count, self.raw_discounts[order]) + removed * '
                      'lower) / total\n'
                      '\n'
                      '    def _log_gram(self, gram):\n'
                      '        probability = self.probability(gram)\n'
                      '        if not 0 < probability <= 1 + 1e-10:\n'
                      '            raise ArithmeticError((repr(gram), probability))\n'
                      '        return math.log(probability)\n'
                      '\n'
                      '    def log_probability(self, sequence, order):\n'
                      '        assert 1 <= order <= self.max_order\n'
                      '        padded = BOS * (order - 1) + sequence + EOS\n'
                      '        grams = Counter(padded[position - order + 1:position + 1] for position in '
                      'range(order - 1, len(padded)))\n'
                      '        return sum(count * self.log_cached(gram) for gram, count in grams.items())\n'
                      '\n'
                      '    def close(self):\n'
                      '        self.lower_cached.cache_clear()\n'
                      '        self.log_cached.cache_clear()',
        'test_modified_kn': 'def test_modified_kn():\n'
                            "    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, "
                            "'e': 1, 'f': 2, 'g': 3, 'h': 1}))\n"
                            '    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value '
                            'in enumerate(values, 1))\n'
                            "    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', "
                            "'abc', 'abc'])\n"
                            "    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6\n"
                            '    for order in [1, 2, 3, 4]:\n'
                            "        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order "
                            '- 1)}\n'
                            '        for context in contexts:\n'
                            '            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 '
                            "else ''\n"
                            '            assert abs(sum(model.probability(history + symbol) for symbol in '
                            'model.support) - 1) < 1e-9, (order, repr(history))\n'
                            "        assert math.isfinite(model.log_probability('a' + UNK, order))\n"
                            "    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)\n"
                            '    model.close()\n'
                            "    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']\n"
                            "    assert tokenize('Hola', 'char')[0] == 'H'\n"
                            "    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, "
                            "BOS/EOS, UNK y capitalizacion correctos.')"},
 'v9': {'inner_split': 'def inner_split(fit_idx):\n'
                       '    fit_groups=np.unique(group_ids[fit_idx])\n'
                       '    '
                       "core_groups,inner_groups=train_test_split(fit_groups,test_size=PARAMS['inner_fraction'],stratify=group_targets[fit_groups],random_state=SEED)\n"
                       '    core_idx=fit_idx[np.isin(group_ids[fit_idx],core_groups)]\n'
                       '    inner_idx=fit_idx[np.isin(group_ids[fit_idx],inner_groups)]\n'
                       '    assert not '
                       'set(train.normalized.iloc[core_idx])&set(train.normalized.iloc[inner_idx])\n'
                       '    assert set(y_all[core_idx])==set(range(6)) and '
                       'set(y_all[inner_idx])==set(range(6))\n'
                       '    return core_idx,inner_idx'}}

In [ ]:
URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)

MENTION_RE = re.compile(r'@\w+', re.UNICODE)

REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

assert ast_gate(current_cell_source(), 'v6', ['URL_RE', 'MENTION_RE', 'REPEAT_RE', 'replace_url', 'normalize_text'])
clean_text = train_raw.text.fillna('').str.strip().map(html.unescape)
excluded = clean_text.eq('') | clean_text.str.contains('#¿NOMBRE', regex=False)
post_mask = clean_text.map(normalize_text)
post_excluded = post_mask.eq('') | post_mask.str.contains('#¿NOMBRE', regex=False)
assert np.array_equal(excluded.to_numpy(), post_excluded.to_numpy()), 'Excluir antes o despues de normalizar cambia las filas; detener.'
if int(excluded.sum()) != EXPECTED_EXCLUDED:
    raise RuntimeError(f'Exclusiones {int(excluded.sum())}; se esperaban {EXPECTED_EXCLUDED}.')
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean_text.loc[~excluded]
train = train.reset_index(drop=True)
train['normalized'] = train.text.map(normalize_text)
y_all = pd.Categorical(train.age_range, categories=CLASSES).codes.astype(np.int64)
NORMALIZED = train.normalized.to_numpy(dtype=object)
TEST_NORMALIZED = test_raw.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy(dtype=object)
log('Train original/limpio/excluidos:', len(train_raw), len(train), int(excluded.sum()))

## 5. Particiones y pliegues (copiados de v6 sin cambios)
Se lee el NPZ de v6 con `allow_pickle=False`. Se comprueba que desarrollo y holdout sean disjuntos y cubran todo el train limpio, y que ningún texto normalizado aparezca a la vez en entrenamiento y validación de un pliegue. Los cinco pliegues exteriores son los de v6: `StratifiedKFold(5, shuffle=True, random_state=42)` sobre los grupos de desarrollo, expandidos a filas.

In [ ]:
assert np.array_equal(split['retained_raw_idx'], train.raw_row.to_numpy()), 'retained_raw_idx no coincide con las filas retenidas.'
group_ids, unique_texts = pd.factorize(train.normalized, sort=True)
assert np.array_equal(group_ids, split['group_ids'])
idx_development, idx_holdout = split['train_idx'], split['valid_idx']
dev_groups, holdout_groups = split['train_groups'], split['valid_groups']
assert not np.intersect1d(idx_development, idx_holdout).size and not np.intersect1d(dev_groups, holdout_groups).size
assert np.array_equal(np.sort(np.concatenate([idx_development, idx_holdout])), np.arange(len(train)))
assert np.array_equal(train.raw_row.to_numpy()[idx_development], split['raw_train_idx'])
assert np.array_equal(train.raw_row.to_numpy()[idx_holdout], split['raw_valid_idx'])
assert not set(NORMALIZED[idx_development]) & set(NORMALIZED[idx_holdout])
group_counts = np.zeros((len(unique_texts), 6), dtype=np.int64)
np.add.at(group_counts, (group_ids, y_all), 1)
group_targets = group_counts.argmax(axis=1)
groups_development = group_ids[idx_development]
group_folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(dev_groups, group_targets[dev_groups]))
folds, fold_rows = [], []
for fold_number, (fit_positions, valid_positions) in enumerate(group_folds, 1):
    fit_idx = idx_development[np.flatnonzero(np.isin(groups_development, dev_groups[fit_positions]))]
    valid_idx = idx_development[np.flatnonzero(np.isin(groups_development, dev_groups[valid_positions]))]
    assert not np.intersect1d(fit_idx, valid_idx).size
    assert not set(NORMALIZED[fit_idx]) & set(NORMALIZED[valid_idx]), ('Fuga de texto normalizado', fold_number)
    folds.append((fit_idx, valid_idx))
    fold_rows.append(dict(pliegue=fold_number, filas_entrenamiento=len(fit_idx), filas_validacion=len(valid_idx)))
display(pd.DataFrame(fold_rows))
observed_sizes = [row['filas_validacion'] for row in fold_rows]
if observed_sizes != EXPECTED_FOLD_SIZES:
    raise RuntimeError(f'Tamanos de validacion {observed_sizes}; se esperaban {EXPECTED_FOLD_SIZES}.')
log('Desarrollo/holdout:', len(idx_development), len(idx_holdout), '| validacion por pliegue:', observed_sizes, '(coinciden con v6)')

## 6. Kneser-Ney modificado (copiado de v6 sin cambios)
Se copian `TOKEN_RE`, `tokenize`, `make_encoder`, `encode_text`, `estimate_discounts`, `discount_for`, `ModifiedKN` y la prueba `test_modified_kn`. Cada clase y cada orden tienen sus propios descuentos D1, D2 y D3+. En el nivel palabra, los tokens con frecuencia menor que 2 en el ajuste pasan a UNK. **Gate AST** contra v6.

In [ ]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

assert ast_gate(current_cell_source(), 'v6', ['TOKEN_RE', 'tokenize', 'make_encoder', 'encode_text', 'estimate_discounts', 'discount_for', 'ModifiedKN', 'test_modified_kn'])
test_modified_kn()

## 7. Raíces con Snowball (nuevo miembro)
Los tokens son los mismos del nivel palabra de v6 (`tokenize(text, 'word')`, en minúsculas). Solo se procesan los tokens alfabéticos (`str.isalpha`): se pasan a minúsculas, se les quitan las tildes con `unicodedata.normalize('NFD')` descartando las marcas `Mn` (igual que en el notebook de análisis del corpus; esto también convierte la ñ en n) y se aplica `SnowballStemmer('spanish').stem`. Números, puntuación, `<url>` y `<user>` se quedan literales.

Cada texto de raíces se forma uniendo las raíces con espacios, y se codifica con las mismas funciones `make_encoder` y `encode_text` del nivel palabra (umbral de frecuencia 2). **Gate:** al volver a tokenizar ese texto tienen que salir exactamente las raíces, en train y en test. Así se comprueba que la codificación no altera la secuencia.

In [ ]:
@lru_cache(maxsize=None)
def root_token(token):
    if not token.isalpha():
        return token
    plain = ''.join(character for character in unicodedata.normalize('NFD', token.lower()) if unicodedata.category(character) != 'Mn')
    return STEMMER.stem(plain)

def root_tokens(text):
    return [root_token(token) for token in tokenize(text, 'word')]

def root_text(text):
    return ' '.join(root_tokens(text))

assert root_tokens('Corriendo con las CANCIONES <url> @x <user> 2026 !!') == ['corr', 'con', 'las', 'cancion', '<url>', '@', 'x', '<user>', '2026', '!', '!']
assert root_token('canción') == root_token('cancion') and root_token('<user>') == '<user>'
ROOT_TEXTS = np.array([root_text(text) for text in NORMALIZED], dtype=object)
TEST_ROOT_TEXTS = np.array([root_text(text) for text in TEST_NORMALIZED], dtype=object)
mismatches = [text for text, rooted in zip(np.concatenate([NORMALIZED, TEST_NORMALIZED]), np.concatenate([ROOT_TEXTS, TEST_ROOT_TEXTS])) if tokenize(rooted, 'word') != root_tokens(text)]
if mismatches:
    raise RuntimeError(f'Gate de raices: {len(mismatches)} textos no se reconstruyen al volver a tokenizar. Ejemplos: {mismatches[:3]}')
LEVEL_TEXTS = {'char': NORMALIZED, 'word': NORMALIZED, 'root': ROOT_TEXTS}
TEST_LEVEL_TEXTS = {'char': TEST_NORMALIZED, 'word': TEST_NORMALIZED, 'root': TEST_ROOT_TEXTS}
with pd.option_context('display.max_colwidth', 120):
    display(pd.DataFrame({'normalizado': NORMALIZED[:5], 'raices': ROOT_TEXTS[:5]}))
log('Raices calculadas: train', len(ROOT_TEXTS), '| test', len(TEST_ROOT_TEXTS), '| tipos distintos en cache', root_token.cache_info().currsize, '| gate de retokenizacion superado')

## 8. Partición interna, caché y ensamble
`inner_split` se copia de v9 (gate AST). Toma el entrenamiento de un pliegue y separa un 10 % de grupos de texto normalizado (*inner*), estratificados por la clase mayoritaria de cada grupo, con `random_state=42`. El resto es el *core*.

- **Score de una clase:** w_car·log P_car + w_pal·log P_pal + w_raiz·log P_raiz + log prior. El prior es la frecuencia de la clase en los datos de ajuste: *core* al puntuar *inner* y el entrenamiento completo del pliegue al puntuar la validación exterior. Se suma una sola vez.
- **Selección:** se elige la combinación con mayor F1 macro sobre *inner*. Si hay empate gana la de mayor w_car y, si sigue el empate, la de mayor w_pal, que es la más cercana a v6.
- **Caché:** cada matriz de scores (modelo × etapa) se guarda en `.npz` con una huella SHA-256 de la configuración, los hashes del train y del NPZ y los índices de ajuste y aplicación. Si la huella no coincide, la ejecución se detiene.

In [ ]:
def inner_split(fit_idx):
    fit_groups=np.unique(group_ids[fit_idx])
    core_groups,inner_groups=train_test_split(fit_groups,test_size=PARAMS['inner_fraction'],stratify=group_targets[fit_groups],random_state=SEED)
    core_idx=fit_idx[np.isin(group_ids[fit_idx],core_groups)]
    inner_idx=fit_idx[np.isin(group_ids[fit_idx],inner_groups)]
    assert not set(train.normalized.iloc[core_idx])&set(train.normalized.iloc[inner_idx])
    assert set(y_all[core_idx])==set(range(6)) and set(y_all[inner_idx])==set(range(6))
    return core_idx,inner_idx

assert ast_gate(current_cell_source(), 'v9', ['inner_split'])
CONFIG = dict(implementation=IMPLEMENTATION_VERSION, seed=SEED, orders=ORDERS, encoder_level=ENCODER_LEVEL, base_epsilon=BASE_EPSILON, simplex=SIMPLEX, inner_fraction=PARAMS['inner_fraction'], stemmer='nltk.stem.snowball.SnowballStemmer(spanish) ' + package_version('nltk'), accents='NFD sin Mn tras minusculas; solo tokens isalpha', train_sha256=TRAIN_SHA, split_sha256=SPLIT_SHA, classes=CLASSES.tolist())
CONFIG_FINGERPRINT = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()

def fingerprint(stage, **items):
    return hashlib.sha256(json.dumps(dict(config=CONFIG_FINGERPRINT, stage=stage, **items), sort_keys=True).encode()).hexdigest()

def index_digest(indices):
    return hashlib.sha256(np.asarray(indices, dtype='<i8').tobytes()).hexdigest()

def save_npz_checked(path, **values):
    temporary = path.with_suffix(path.suffix + '.partial')
    with temporary.open('wb') as handle:
        np.savez_compressed(handle, **values)
    temporary.replace(path)

def load_npz_checked(path, signature):
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as handle:
        payload = {name: handle[name].copy() for name in handle.files}
    if str(payload['fingerprint']) != signature:
        raise RuntimeError(f'Cache incompatible en {path}: la huella no coincide. Se detiene en lugar de reutilizarla. Mueva o borre esa carpeta si quiere recomputar.')
    return payload

def save_json_new_or_same(path, payload):
    text = json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True)
    if path.exists():
        stored = json.loads(path.read_text(encoding='utf-8'))
        if stored.get('fingerprint') != payload['fingerprint']:
            raise RuntimeError(f'{path} tiene otra huella; se detiene.')
        if stored != json.loads(text):
            raise RuntimeError(f'{path} existe con la misma huella pero otros valores: el calculo no es reproducible; se detiene.')
        return
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(text, encoding='utf-8')
    temporary.replace(path)

def macro_f1(targets, predictions):
    return float(f1_score(targets, predictions, labels=np.arange(6), average='macro', zero_division=0))

def log_prior(indices):
    return np.log(np.bincount(y_all[indices], minlength=6) / len(indices))

def combine(scores, weights, prior):
    weight_char, weight_word, weight_root = weights
    return weight_char * scores['char'] + weight_word * scores['word'] + weight_root * scores['root'] + prior

def score_level(level, fit_texts, fit_labels, app_texts, path, signature):
    cached = load_npz_checked(path, signature)
    if cached is not None:
        assert cached['scores'].shape == (len(app_texts), 6)
        return cached['scores'], dict(vocabulario=int(cached['vocab_size']), segundos=float(cached['seconds']), reutilizado=True)
    started = time.perf_counter()
    encoder_level = ENCODER_LEVEL[level]
    mapping = make_encoder(fit_texts, encoder_level)
    encoded_fit = [encode_text(text, encoder_level, mapping) for text in fit_texts]
    encoded_app = [encode_text(text, encoder_level, mapping) for text in app_texts]
    scores = np.zeros((len(app_texts), 6), dtype=np.float64)
    for class_index in range(6):
        model = ModifiedKN(ORDERS[level], set(mapping.values())).fit(sequence for sequence, target in zip(encoded_fit, fit_labels) if target == class_index)
        scores[:, class_index] = [model.log_probability(sequence, ORDERS[level]) for sequence in encoded_app]
        model.close()
        del model
        gc.collect()
    assert np.isfinite(scores).all()
    seconds = time.perf_counter() - started
    save_npz_checked(path, fingerprint=np.array(signature), scores=scores, vocab_size=np.array(len(mapping)), seconds=np.array(seconds))
    return scores, dict(vocabulario=len(mapping), segundos=seconds, reutilizado=False)

def select_weights(inner_scores, inner_targets, prior):
    rows = [dict(w_car=weights[0], w_pal=weights[1], w_raiz=weights[2], f1_interno=macro_f1(inner_targets, combine(inner_scores, weights, prior).argmax(axis=1))) for weights in SIMPLEX]
    table = pd.DataFrame(rows).sort_values(['f1_interno', 'w_car', 'w_pal'], ascending=False, kind='mergesort').reset_index(drop=True)
    table.insert(0, 'rango', np.arange(1, len(table) + 1))
    chosen = tuple(float(value) for value in table.loc[0, ['w_car', 'w_pal', 'w_raiz']])
    return table, chosen

def disjoint_gate(core_idx, inner_idx, app_idx=None):
    parts = {'core': core_idx, 'inner': inner_idx}
    if app_idx is not None:
        parts['exterior'] = app_idx
    names = list(parts)
    for position, first in enumerate(names):
        for second in names[position + 1:]:
            if np.intersect1d(parts[first], parts[second]).size or set(NORMALIZED[parts[first]]) & set(NORMALIZED[parts[second]]):
                raise RuntimeError(f'Gate de disjuncion: {first} y {second} comparten filas o textos normalizados.')
    return True

def run_procedure(stage_dir, fit_idx, app_texts, app_key, app_idx=None):
    stage_dir.mkdir(parents=True, exist_ok=True)
    core_idx, inner_idx = inner_split(fit_idx)
    disjoint_gate(core_idx, inner_idx, app_idx)
    inner_scores, outer_scores, info = {}, {}, []
    for level in MODELS:
        inner_scores[level], details = score_level(level, LEVEL_TEXTS[level][core_idx], y_all[core_idx], LEVEL_TEXTS[level][inner_idx], stage_dir / f'interno_{MODEL_NAMES[level]}.npz', fingerprint('interno', level=level, fit=index_digest(core_idx), app=index_digest(inner_idx)))
        info.append(dict(etapa='core->inner', modelo=MODEL_NAMES[level], **details))
    table, chosen = select_weights(inner_scores, y_all[inner_idx], log_prior(core_idx))
    for level in MODELS:
        outer_scores[level], details = score_level(level, LEVEL_TEXTS[level][fit_idx], y_all[fit_idx], app_texts[level], stage_dir / f'exterior_{MODEL_NAMES[level]}.npz', fingerprint('exterior', level=level, fit=index_digest(fit_idx), app=app_key))
        info.append(dict(etapa='fit->exterior', modelo=MODEL_NAMES[level], **details))
    table.to_csv(stage_dir / 'combinaciones_internas.csv', index=False)
    return dict(core_idx=core_idx, inner_idx=inner_idx, table=table, chosen=chosen, outer_scores=outer_scores, prior=log_prior(fit_idx), info=pd.DataFrame(info))

def token_statistics(indices):
    counts = Counter()
    for text in NORMALIZED[indices]:
        counts.update(tokenize(text, 'word'))
    roots = Counter()
    for token, count in counts.items():
        roots[root_token(token)] += count
    total = sum(counts.values())
    changed = sum(count for token, count in counts.items() if root_token(token) != token)
    return dict(tokens=total, tokens_cambiados_pct=100 * changed / total, tipos_palabra=len(counts), tipos_raiz=len(roots), vocab_palabra_min2=sum(count >= 2 for count in counts.values()), vocab_raiz_min2=sum(count >= 2 for count in roots.values()))

if any(abs(sum(weights) - 1) > 1e-12 for weights in SIMPLEX) or len(SIMPLEX) != 15 or V6_WEIGHTS not in SIMPLEX:
    raise RuntimeError('Gate de pesos fallido: el simplex no suma 1, no tiene 15 combinaciones o no incluye v6.')
display(pd.DataFrame(SIMPLEX, columns=['w_car', 'w_pal', 'w_raiz']).assign(suma=lambda frame: frame.sum(axis=1)))
log('Gate de pesos superado: 15 combinaciones, suma 1 con tolerancia 1e-12, incluye v6', V6_WEIGHTS, '| huella de configuracion', CONFIG_FINGERPRINT[:16])

## 9. Guardia de cumplimiento
Recorre con AST todo el código ejecutado en la sesión y se detiene si aparecen vectorizadores, embeddings, Word2Vec/GloVe, LSTM/GRU, transformers, cargas preentrenadas, spaCy o lematizadores. La única importación de NLTK permitida es `from nltk.stem.snowball import SnowballStemmer`, y de torch solo puede usarse `manual_seed`.

In [ ]:
def compliance_guard(sources):
    forbidden = {'tfidfvectorizer', 'tfidftransformer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'onehotencoder', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove', 'fasttext', 'gensim', 'lstm', 'gru', 'nn', 'spacy', 'stanza', 'lemmatizer', 'wordnetlemmatizer', 'lemmatize', 'from_pretrained', 'automodel', 'autotokenizer', 'sentence_transformers'}
    findings = []
    for block_number, source in enumerate(sources, 1):
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
                if isinstance(node.value, ast.Name) and node.value.id == 'torch' and node.attr not in {'manual_seed', '__version__'}:
                    findings.append((block_number, node.lineno, 'torch.' + node.attr))
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
                findings.extend((block_number, node.lineno, alias.name) for alias in node.names if alias.name.split('.')[0] == 'nltk')
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
                if (node.module or '').split('.')[0] == 'nltk' and (node.module != 'nltk.stem.snowball' or [alias.name for alias in node.names] != ['SnowballStemmer']):
                    findings.append((block_number, node.lineno, f'nltk no permitido: {node.module}'))
            for name in names:
                if forbidden.intersection(part.lower() for part in name.split('.')):
                    findings.append((block_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                findings.append((block_number, node.lineno, 'ejecucion dinamica'))
    if findings:
        raise RuntimeError(f'Guardia de cumplimiento: usos prohibidos {findings}')
    return True

executed_sources = [text for text in get_ipython().history_manager.input_hist_raw if text.strip()]
assert compliance_guard(executed_sources)
log('Guardia de cumplimiento superada sobre', len(executed_sources), 'celdas ejecutadas.')

## 10. Validación cruzada exterior (cinco pliegues de v6)
En cada pliegue: (1) `inner_split`; (2) los tres modelos por clase se ajustan con *core* y puntúan *inner*; (3) se elige la combinación del simplex; (4) los tres modelos se reajustan con el 100 % del entrenamiento del pliegue y puntúan la validación exterior. Esa validación nunca interviene en la elección.

Los artefactos se guardan en `fold_k/`: scores exteriores e internos de cada modelo, `combinaciones_internas.csv`, `resultado.json` y las predicciones. **Gate de reproducción:** en el pliegue 1, la combinación (0,75; 0,25; 0) tiene que dar F1 = 0,443844 ± 1e-6. Si no, la ejecución se detiene antes de los pliegues 2 a 5.

In [ ]:
cv_rows, diagnostic_rows, token_rows, fold_outputs = [], [], [], {}
oof_v6 = np.full(len(train), -1, dtype=np.int64)
oof_v10 = np.full(len(train), -1, dtype=np.int64)
cv_started = time.perf_counter()
for fold_number, (fit_idx, valid_idx) in enumerate(folds, 1):
    started = time.perf_counter()
    fold_dir = OUT_DIR / f'fold_{fold_number}'
    previously_done = (fold_dir / 'resultado.json').exists()
    log(f'--- Pliegue {fold_number}:', 'completado antes; se recupera de cache' if previously_done else 'calculando', '| entrenamiento', len(fit_idx), '| validacion', len(valid_idx))
    outcome = run_procedure(fold_dir, fit_idx, {level: LEVEL_TEXTS[level][valid_idx] for level in MODELS}, index_digest(valid_idx), valid_idx)
    targets = y_all[valid_idx]
    pred_v6 = combine(outcome['outer_scores'], V6_WEIGHTS, outcome['prior']).argmax(axis=1)
    pred_v10 = combine(outcome['outer_scores'], outcome['chosen'], outcome['prior']).argmax(axis=1)
    single = {level: combine(outcome['outer_scores'], tuple(1.0 if other == level else 0.0 for other in MODELS), outcome['prior']).argmax(axis=1) for level in MODELS}
    f1_v6, f1_v10 = macro_f1(targets, pred_v6), macro_f1(targets, pred_v10)
    if fold_number == 1:
        gap = abs(f1_v6 - V6_FOLD1_F1)
        log(f'Gate de reproduccion pliegue 1: F1 (0.75, 0.25, 0) = {f1_v6:.7f}; referencia v6 {V6_FOLD1_F1}; diferencia {gap:.2e}')
        if gap > GATE_TOLERANCE:
            raise RuntimeError(f'Gate de reproduccion fallido: {f1_v6:.7f} frente a {V6_FOLD1_F1} (tolerancia {GATE_TOLERANCE}). La reimplementacion no es identica a v6; se detiene.')
    oof_v6[valid_idx], oof_v10[valid_idx] = pred_v6, pred_v10
    stats = token_statistics(fit_idx)
    vocab = outcome['info'].set_index(['etapa', 'modelo']).vocabulario
    stats.update(pliegue=fold_number, vocab_modelo_palabras=int(vocab[('fit->exterior', 'palabras')]), vocab_modelo_raices=int(vocab[('fit->exterior', 'raices')]))
    token_rows.append(stats)
    result = dict(fingerprint=fingerprint('resultado_pliegue', fold=fold_number, fit=index_digest(fit_idx), valid=index_digest(valid_idx)), pliegue=fold_number, combinacion_elegida=list(outcome['chosen']), f1_interno_elegida=float(outcome['table'].f1_interno.iloc[0]), f1_v6=f1_v6, f1_v10=f1_v10, diferencia=f1_v10 - f1_v6, **{f'f1_solo_{MODEL_NAMES[level]}': macro_f1(targets, single[level]) for level in MODELS})
    save_json_new_or_same(fold_dir / 'resultado.json', result)
    save_npz_checked(fold_dir / 'predicciones_exterior.npz', fingerprint=np.array(result['fingerprint']), indices=valid_idx, y=targets, pred_v6=pred_v6, pred_v10=pred_v10, **{f'pred_{MODEL_NAMES[level]}': single[level] for level in MODELS})
    cv_rows.append(dict(pliegue=fold_number, f1_v6=f1_v6, f1_v10=f1_v10, diferencia=f1_v10 - f1_v6, combinacion=outcome['chosen']))
    diagnostic_rows.append(dict(pliegue=fold_number, **{f'f1_{MODEL_NAMES[level]}': result[f'f1_solo_{MODEL_NAMES[level]}'] for level in MODELS}, f1_v6=f1_v6, f1_v10=f1_v10, w_car=outcome['chosen'][0], w_pal=outcome['chosen'][1], w_raiz=outcome['chosen'][2], f1_interno=result['f1_interno_elegida']))
    fold_outputs[fold_number] = dict(table=outcome['table'], targets=targets, pred_v10=pred_v10)
    display(outcome['info'])
    display(outcome['table'].head(5))
    log(f'Pliegue {fold_number}: F1 v6 {f1_v6:.6f} | F1 v10 {f1_v10:.6f} | diferencia {f1_v10 - f1_v6:+.6f} | combinacion {outcome["chosen"]} | {time.perf_counter() - started:.1f} s | {memory_report()}')
    del outcome
    gc.collect()
assert (oof_v6[idx_development] >= 0).all() and (oof_v10[idx_development] >= 0).all()
log(f'CV completa en {time.perf_counter() - cv_started:.1f} s')

## 11. Métricas y decisión
La regla es la de v6 y se fijó antes de ver los resultados: **PASA** si la media de las diferencias pareadas es positiva y mayor que su desviación muestral (ddof=1).

In [ ]:
cv_table = pd.DataFrame(cv_rows)
difference_mean = float(cv_table.diferencia.mean())
difference_std = float(cv_table.diferencia.std(ddof=1))
PASA = bool(difference_mean > 0 and difference_mean > difference_std)
oof_f1_v6 = macro_f1(y_all[idx_development], oof_v6[idx_development])
oof_f1_v10 = macro_f1(y_all[idx_development], oof_v10[idx_development])
display(cv_table)
cv_table.assign(combinacion=cv_table.combinacion.map(str)).to_csv(OUT_DIR / 'tabla_cv.csv', index=False)
cv_metrics = dict(fingerprint=fingerprint('metricas_cv'), pliegues=cv_table.assign(combinacion=cv_table.combinacion.map(list)).to_dict(orient='records'), media_diferencias=difference_mean, desviacion_muestral_ddof1=difference_std, decision='PASA' if PASA else 'NO PASA', f1_oof_v6=oof_f1_v6, f1_oof_v10=oof_f1_v10)
save_json_new_or_same(OUT_DIR / 'metricas_cv.json', cv_metrics)
log(f'Media de diferencias {difference_mean:+.6f} | desviacion muestral (ddof=1) {difference_std:.6f} | F1 OOF v6 {oof_f1_v6:.6f} v10 {oof_f1_v10:.6f}')
log('DECISION:', 'PASA' if PASA else 'NO PASA')

## 12. Diagnóstico (se calcula aunque no pase)

In [ ]:
diagnostic_table = pd.DataFrame(diagnostic_rows)
print('F1 macro por pliegue: cada modelo solo (con log prior), v6 y v10, y pesos elegidos')
display(diagnostic_table)
display(diagnostic_table.drop(columns='pliegue').agg(['mean', 'std']))
token_table = pd.DataFrame(token_rows)[['pliegue', 'tokens', 'tokens_cambiados_pct', 'tipos_palabra', 'tipos_raiz', 'vocab_palabra_min2', 'vocab_raiz_min2', 'vocab_modelo_palabras', 'vocab_modelo_raices']]
assert (token_table.vocab_palabra_min2 == token_table.vocab_modelo_palabras).all() and (token_table.vocab_raiz_min2 == token_table.vocab_modelo_raices).all()
print('Tokens que cambian al aplicar la raiz (incluye cambios solo por tildes) y vocabulario del entrenamiento de cada pliegue')
display(token_table)
chosen_set = sorted(set(map(tuple, diagnostic_table[['w_car', 'w_pal', 'w_raiz']].to_numpy().tolist())))
log('Pesos elegidos por pliegue:', diagnostic_table[['w_car', 'w_pal', 'w_raiz']].to_numpy().tolist(), '| siempre la misma combinacion' if len(chosen_set) == 1 else f'| {len(chosen_set)} combinaciones distintas')
print('Tabla interna de las 15 combinaciones, pliegue 1')
display(fold_outputs[1]['table'])
print('Matriz de confusion del ensamble v10, pliegue 1 (filas reales, columnas predichas)')
display(pd.DataFrame(confusion_matrix(fold_outputs[1]['targets'], fold_outputs[1]['pred_v10'], labels=np.arange(6)), index=pd.Index(CLASSES, name='real'), columns=pd.Index(CLASSES, name='predicha')))
distribution = pd.DataFrame({'real': np.bincount(y_all[idx_development], minlength=6), 'pred_v6': np.bincount(oof_v6[idx_development], minlength=6), 'pred_v10': np.bincount(oof_v10[idx_development], minlength=6)}, index=CLASSES)
print('Distribucion de predicciones por clase (OOF, cinco pliegues)')
display(distribution)
diagnostic_table.to_csv(OUT_DIR / 'diagnostico_pliegues.csv', index=False)
token_table.to_csv(OUT_DIR / 'diagnostico_raices.csv', index=False)
distribution.to_csv(OUT_DIR / 'distribucion_predicciones.csv')

## 13. Holdout (solo si PASA; una sola vez)
Se aplica el mismo procedimiento que en cada pliegue, ahora sobre todo el desarrollo (`train_idx`): partición interna, elección de pesos en *inner* y reajuste con el desarrollo completo. Después se puntúa el holdout. Si `holdout_metrics.json` ya existe, se lee y no se vuelve a evaluar.

In [ ]:
if PASA:
    assert 'holdout_f1' not in globals(), 'El holdout ya se consulto en este kernel.'
    metrics_path = OUT_DIR / 'holdout_metrics.json'
    signature = fingerprint('holdout', fit=index_digest(idx_development), valid=index_digest(idx_holdout))
    if metrics_path.exists():
        holdout_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        if holdout_metrics['fingerprint'] != signature:
            raise RuntimeError('holdout_metrics.json tiene otra huella; se detiene sin reevaluar.')
        log('Holdout evaluado en una ejecucion anterior: se recupera y NO se vuelve a evaluar.')
    else:
        outcome = run_procedure(OUT_DIR / 'holdout', idx_development, {level: LEVEL_TEXTS[level][idx_holdout] for level in MODELS}, index_digest(idx_holdout), idx_holdout)
        holdout_pred = combine(outcome['outer_scores'], outcome['chosen'], outcome['prior']).argmax(axis=1)
        y_holdout = y_all[idx_holdout]
        holdout_metrics = dict(fingerprint=signature, combinacion=list(outcome['chosen']), f1_macro=macro_f1(y_holdout, holdout_pred), reporte=classification_report(y_holdout, holdout_pred, labels=np.arange(6), target_names=CLASSES, output_dict=True, zero_division=0), matriz_confusion=confusion_matrix(y_holdout, holdout_pred, labels=np.arange(6)).tolist())
        save_npz_checked(OUT_DIR / 'holdout' / 'predicciones_holdout.npz', fingerprint=np.array(signature), indices=idx_holdout, predictions=holdout_pred)
        with metrics_path.open('x', encoding='utf-8') as handle:
            json.dump(holdout_metrics, handle, ensure_ascii=False, indent=2)
    holdout_f1 = float(holdout_metrics['f1_macro'])
    log(f'F1 macro holdout v10: {holdout_f1:.6f} | combinacion {holdout_metrics["combinacion"]}')
    display(pd.DataFrame(holdout_metrics['reporte']).T.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']])
    display(pd.DataFrame(holdout_metrics['matriz_confusion'], index=pd.Index(CLASSES, name='real'), columns=pd.Index(CLASSES, name='predicha')))
else:
    log('HOLDOUT NO EVALUADO: la regla no se cumple.')

## 14. Ajuste final y submission (solo si PASA)
Se repite el procedimiento con el train completo: partición interna, elección de pesos y reajuste. Después se puntúa `age_test.csv`. La submission respeta las columnas y el orden de `age_submission.csv`. Si ya existe un archivo con contenido distinto, no se sobrescribe y la ejecución se detiene. No se envía nada a Kaggle.

In [ ]:
if PASA:
    all_idx = np.arange(len(train))
    outcome = run_procedure(OUT_DIR / 'final', all_idx, TEST_LEVEL_TEXTS, TEST_SHA)
    final_pred = combine(outcome['outer_scores'], outcome['chosen'], outcome['prior']).argmax(axis=1)
    assert len(final_pred) == len(test_raw)
    predicted = pd.DataFrame({'id': test_raw.id, 'age_range': CLASSES[final_pred]}).set_index('id')
    submission = predicted.loc[sample.id].reset_index()[sample.columns.tolist()]
    assert submission.columns.tolist() == ['id', 'age_range'] and submission.id.tolist() == sample.id.tolist()
    assert submission.age_range.isin(CLASSES).all() and len(submission) == len(sample)
    payload = submission.to_csv(index=False, sep=',').encode('utf-8')
    if SUBMISSION_PATH.exists():
        if SUBMISSION_PATH.read_bytes() != payload:
            raise RuntimeError(f'{SUBMISSION_PATH} ya existe con contenido distinto; no se sobrescribe.')
        log('Submission ya existente e identica:', SUBMISSION_PATH)
    else:
        with SUBMISSION_PATH.open('xb') as handle:
            handle.write(payload)
        log('Submission escrita:', SUBMISSION_PATH, '| filas', len(submission), '| combinacion', outcome['chosen'])
    display(submission.head())
    display(submission.age_range.value_counts().reindex(CLASSES).rename('predicciones').to_frame())
else:
    log('SUBMISSION NO GENERADA: no se entrena con el train completo.')

## 15. Entrega

In [ ]:
final_table = cv_table.assign(combinacion=cv_table.combinacion.map(lambda weights: f'({weights[0]:.2f}; {weights[1]:.2f}; {weights[2]:.2f})')).rename(columns={'f1_v6': 'F1 v6', 'f1_v10': 'F1 v10', 'combinacion': 'combinación elegida (car; pal; raíz)'})
display(final_table.style.format({'F1 v6': '{:.6f}', 'F1 v10': '{:.6f}', 'diferencia': '{:+.6f}'}))
display(Markdown(f"""**Media de diferencias:** {difference_mean:+.6f} · **desviación muestral (ddof=1):** {difference_std:.6f} · **Decisión: {'PASA' if PASA else 'NO PASA'}**

F1 OOF en desarrollo: v6 {oof_f1_v6:.6f} · v10 {oof_f1_v10:.6f}. {'Holdout v10: ' + format(holdout_f1, '.6f') if PASA else 'Holdout no evaluado; submission no generada.'}"""))
log('Fin. Artefactos en', OUT_DIR, '| No se envio nada a Kaggle | No se modifico v6.')

## Notas sobre la interpretación de las reglas
Las reglas citadas están en `claude/DECISION_REGLAS_TWEETAGE.md`. Como ese documento no estaba en el repositorio al crear el notebook, las interpretaciones siguientes parten del pedido y del código de v6/v9.

1. **Raíces y "vectorización".** Snowball es un algoritmo de reglas sin parámetros aprendidos. Las raíces solo cambian los símbolos que modela KN; no se construyen bolsas, conteos como features ni vectores. Los conteos de n-gramas internos de KN son el propio modelo de lenguaje, igual que en v6.
2. **Codificación de raíces.** Se reutilizan `make_encoder` y `encode_text` del nivel palabra sobre el texto de raíces unido con espacios. El gate de retokenización comprueba que la secuencia sea exactamente la de las raíces. El umbral UNK (frecuencia < 2) es el de v6.
3. **Quitar tildes.** Se usa NFD y se descartan las marcas `Mn`, como en el notebook de análisis del corpus. En consecuencia, la ñ pasa a n antes de aplicar la raíz. El porcentaje de tokens que cambian incluye los que solo pierden la tilde.
4. **Prior.** Al puntuar *inner* se usa la frecuencia de cada clase en *core*, que son los datos de ajuste de esos modelos. Al puntuar la validación exterior se usa la del entrenamiento completo del pliegue. En los dos casos se suma una sola vez.
5. **Empates.** Gana la combinación con mayor w_car. Si sigue el empate, la de mayor w_pal: así se elige la más cercana a v6 (w_raiz = 0).
6. **Holdout y test.** El pedido no fija los pesos para estas etapas, así que se aplica el mismo procedimiento evaluado en CV (elegir en *inner* y reajustar con todo). Los pesos no se fijan a partir de la CV exterior.
7. **Torch.** Solo se importa para `torch.manual_seed(42)`. La guardia impide cualquier otro uso.
8. **Submission.** Se escribe en `RESULTS/tweetage_v10_submission.csv` (sección 7 del pedido), fuera de `tweetage_v10_raices/`. La ruta está en la celda de configuración.